# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**Method:** this cell makes sure DuckDB is version 1.1 or newer, the first version whose Hugging Face login can read the token from the environment.


In [ ]:
%pip -q install "duckdb>=1.1"

## Setup

**Method, step by step:**

1. **Finds the project folder** and **reads the Hugging Face token**, as in the data contract: environment variable or `.env` file, then a Colab Secret named `HF_TOKEN`, then a typed prompt as the last resort. DuckDB reads the token from the environment, so it never appears in a query or an output.
2. **Loads the tools.** `duckdb` queries the warehouse, `pandas` and `numpy` handle the results, `scipy` supplies the negative binomial luck test behind the target and rule B, `scikit-learn` supplies logistic regression, the random forest used in the paper audit, the fold splitters, and the metrics, and `matplotlib` draws the one chart.
3. **Defines the months and the cache.** The same three development pairs as the data contract; June 2026 is never read. Heavy queries go through `cached()`, which saves each result once under `work/outputs/w03_cache/` and reuses it on later runs.
4. **Defines `show()` and `show_table()`**, which display every result as a titled table.


In [ ]:
import getpass
import os
import re
import sys
import tempfile
import urllib.request

import duckdb
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from matplotlib import font_manager
from scipy.stats import nbinom
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, r2_score, roc_auc_score
from sklearn.model_selection import GroupKFold, KFold
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler

# Locally, walk up to the project's top folder (the one holding data/raw and .env); Colab keeps its own folder.
if "google.colab" not in sys.modules:
    while not os.path.isdir("data/raw") and os.path.dirname(os.getcwd()) != os.getcwd():
        os.chdir("..")


def load_env_file(path: str = ".env") -> None:
    """Copy KEY=value lines from a local .env file into the environment, keeping any value already set."""
    if not os.path.exists(path):
        return
    with open(path, encoding="utf-8-sig") as handle:
        for line in handle:
            key, sep, value = line.strip().partition("=")
            if sep and key and not key.startswith("#"):
                os.environ.setdefault(key.strip(), value.strip())


def read_colab_secret(name: str) -> str | None:
    """Return the Colab Secret called `name`, or None when it cannot be read.

    Outside Colab the import fails. On Colab the secret can be missing, not shared with this
    notebook, or the request can time out. Each case returns None so the caller can ask for the token.
    """
    try:
        from google.colab import userdata
    except ImportError:
        return None
    try:
        return userdata.get(name)
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError, userdata.TimeoutException):
        return None


# Token order: environment variable or .env file, then Colab Secret, then a typed prompt as the last resort.
load_env_file()
token_source = "environment variable or .env file"
if not os.environ.get("HF_TOKEN"):
    secret = read_colab_secret("HF_TOKEN")
    token_source = "Colab Secret" if secret else "typed prompt"
    os.environ["HF_TOKEN"] = secret or getpass.getpass("Hugging Face READ token (hf_...): ")

# DuckDB reads the token from the environment, so it never appears in a query or an output.
con = duckdb.connect()
con.execute("CREATE OR REPLACE SECRET hf (TYPE huggingface, PROVIDER credential_chain)")
# Cap DuckDB's memory so large scans spill to disk, and hide its progress bars, which clutter saved outputs.
con.execute("SET memory_limit = '2GB'")
con.execute("SET preserve_insertion_order = false")
con.execute("SET enable_progress_bar = false")

REL = "hf://datasets/FlyRank/internship-warehouse"
DIM_CLIENTS = f"read_parquet('{REL}/dim_clients.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"
QUERY_90D = f"read_parquet('{REL}/fact_content_query_90d.parquet')"
DAILY_FOOTERS = f"read_parquet('{REL}/fact_content_daily_performance/*/*.parquet')"
PAIRS = [("2026-02", "2026-03"), ("2026-03", "2026-04"), ("2026-04", "2026-05")]
DEV_MONTHS = sorted({month for pair in PAIRS for month in pair})
PANEL_MONTHS = list(pd.period_range("2025-01", "2026-05", freq="M").astype(str))
assert "2026-06" not in DEV_MONTHS + PANEL_MONTHS, "The sealed month must never be read."


def month_files(months: list[str]) -> str:
    """Return a DuckDB source that reads exactly the listed monthly partitions of the daily table."""
    paths = ", ".join(f"'{REL}/fact_content_daily_performance/month={month}/*.parquet'" for month in months)
    return f"read_parquet([{paths}])"


CACHE = "work/outputs/w03_cache"
os.makedirs(CACHE, exist_ok=True)


def cached(name: str, sql: str) -> pd.DataFrame:
    """Run a warehouse query once, save its result as Parquet in the cache folder, and reuse it afterwards.

    Repeated scans of the hosted files are slow and can hit rate limits, so every heavy query goes
    through here. Delete the cache folder to force a fresh scan.
    """
    path = f"{CACHE}/{name}.parquet"
    if not os.path.exists(path):
        # Write under a temporary name first, so an interrupted run never leaves a half-written cache file.
        con.execute(f"COPY ({sql}) TO '{path}.tmp' (FORMAT parquet)")
        os.replace(f"{path}.tmp", path)
    return con.sql(f"SELECT * FROM read_parquet('{path}')").df()


NUMBER = r"^[-+]?[\d,]*\.?\d+(?:%|x)?$"


def show_table(title: str, frame: pd.DataFrame) -> None:
    """Display a DataFrame as a titled Markdown table.

    Markdown tables look the same in Jupyter, VS Code, Colab, and GitHub, and they avoid the extra
    index row and interactive-table scripts that a plain DataFrame display adds. Columns whose
    cells are all numbers are right-aligned.
    """
    body = frame.reset_index() if frame.index.name or isinstance(frame.index, pd.MultiIndex) else frame
    cells = body.astype(str).apply(lambda col: col.str.replace("|", "\\|", regex=False).str.replace("\n", " ", regex=False))
    align = ["---:" if cells[name].str.strip().str.match(NUMBER).all() else "---" for name in cells.columns]
    lines = ["| " + " | ".join(map(str, cells.columns)) + " |", "| " + " | ".join(align) + " |"]
    lines += ["| " + " | ".join(row) + " |" for row in cells.itertuples(index=False)]
    display(Markdown(f"**{title}**\n\n" + "\n".join(lines)))


def show(title: str, rows: dict[str, object]) -> None:
    """Display a titled two-column table of measures and their values."""
    show_table(title, pd.DataFrame({"Measure": list(rows), "Value": [str(value) for value in rows.values()]}))


show("Connected to the warehouse", {
    "token read from": token_source,
    "token shown anywhere": "no (DuckDB reads it from the environment)",
    "development pairs": ", ".join(f"{a} → {b}" for a, b in PAIRS),
    "sealed pair, never read": "2026-05 → 2026-06",
    "cache folder": CACHE,
})

## Data

The audit uses the same three development pairs, eligibility rules, spread factor, target, and 25 features as the modelling notebook. The sealed May→June pair is added only in section 2b.


**Method:** this cell loads the page-month table for February to May 2026 with the same query and cache as the earlier notebooks, each client's search-history start date, and, from the content table, the content type, the keyword-research fields, and the latest optimization date.


In [ ]:
# The same page-month query and cache as the earlier notebooks.
PAGE_MONTH_SQL = """
    SELECT '{month}' AS month, client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS impressions,
           SUM(gsc_clicks) AS clicks,
           SUM(CASE WHEN gsc_avg_position >= 1 THEN gsc_impressions ELSE 0 END) AS position_impressions,
           SUM(CASE WHEN gsc_avg_position >= 1 THEN gsc_sum_position ELSE 0 END) AS sum_position,
           SUM(CASE WHEN gsc_avg_position >= 1 THEN gsc_sum_position * gsc_avg_position ELSE 0 END) AS sum_position_sq,
           SUM(CASE WHEN gsc_avg_position >= 1 THEN 0 ELSE gsc_clicks END) AS clicks_without_position,
           COUNT(*) AS days_with_impressions,
           SUM(CASE WHEN ga4_data_available IS TRUE THEN ga4_sessions ELSE 0 END) AS ga4_sessions,
           SUM(CASE WHEN ga4_data_available IS TRUE THEN ga4_engaged_sessions ELSE 0 END) AS ga4_engaged_sessions
    FROM {source}
    WHERE gsc_impressions > 0
    GROUP BY client_hash_id, content_hash_id"""
page_months = pd.concat([
    cached(f"page_month_v3_{month}", PAGE_MONTH_SQL.format(month=month, source=month_files([month]))) for month in DEV_MONTHS
], ignore_index=True)
assert not page_months.duplicated(["month", "content_hash_id"]).any(), "A page appears twice in one month."

clients = con.sql(f"SELECT client_hash_id, gsc_data_start FROM {DIM_CLIENTS}").df()
clients["gsc_data_start"] = pd.to_datetime(clients["gsc_data_start"])
content = con.sql(f"SELECT content_hash_id, content_type, main_intent, search_volume, competition, cpc, keyword_token_count, last_optimized_date FROM {DIM_CONTENT}").df()
show_table("Inputs loaded", pd.DataFrame({
    "Rows": [f"{len(page_months):,}", f"{len(clients):,}", f"{len(content):,}"],
    "What it is": ["Page-months, February to May 2026", "Clients with their search-history start date",
                   "Pages with content type, keyword research fields, and latest optimization date"],
}, index=pd.Index(["page_months", "clients", "content"], name="Table")))

**Method:** this cell applies the data contract's rules to each development pair (impressions in both months, client history covering all of month M, a valid position in both months, not optimized during the pair). For each month it computes the impression-weighted position, its daily spread, expected clicks from the fine position bins and from the five tiers (both client-scaled), and whether the page fell short by the locked rule: clicks at most half of expected, with negative binomial p below 0.05. The spread factor α is estimated exactly as the data contract does (one estimate per pair from pages whose position barely moved, the median rounded to three decimals), so every notebook shares one target. The month M+1 version of the rule is the target.


In [ ]:
# Rules and bins exactly as in the data contract; the spread factor is estimated below with its method.
BIN_EDGES = [1.5, 2.5, 3.5, 4.5, 5.5, 6.5, 7.5, 8.5, 9.5, 10.5, 20.5, 50.5]
BIN_NAMES = [str(position) for position in range(1, 11)] + ["11-20", "21-50", "50+"]


def position_bin(position: pd.Series) -> pd.Series:
    """Return each page's position bin: exact rank up to 10, then 11-20, 21-50, and 50+."""
    return pd.Series(np.array(BIN_NAMES)[np.searchsorted(BIN_EDGES, position, side="right")], index=position.index)


def position_tier(position: pd.Series) -> pd.Series:
    """Return the five tiers used by the earlier score: below 3, below 10, below 20, below 50, and 50 or beyond."""
    tiers = np.select([position < 3, position < 10, position < 20, position < 50], ["top_3", "page_1", "striking", "page_3_5"], "deep")
    return pd.Series(tiers, index=position.index)


def client_aware_expected(frame: pd.DataFrame, clicks: str, impressions: str, group: pd.Series) -> pd.Series:
    """Return expected clicks from each group's pooled click rate, scaled by the page's client factor (+1 on both sides)."""
    rate = frame[clicks].groupby(group).sum() / frame[impressions].groupby(group).sum()
    expected = frame[impressions] * group.map(rate)
    by_client = frame.assign(_expected=expected).groupby("client_hash_id")
    factor = (by_client[clicks].sum() + 1) / (by_client["_expected"].sum() + 1)
    return expected * frame["client_hash_id"].map(factor)


def covered_clients(month_m: str) -> pd.Series:
    """Return the clients whose search history starts on or before the first day of month M."""
    return clients.loc[clients["gsc_data_start"] <= pd.Timestamp(f"{month_m}-01"), "client_hash_id"]


def add_month_measures(rows: pd.DataFrame, suffix: str) -> pd.DataFrame:
    """Add position, its spread, click-through, and both expectations for one month."""
    impressions, clicks = rows[f"impressions_{suffix}"], rows[f"clicks_{suffix}"]
    rows[f"position_{suffix}"] = rows[f"sum_position_{suffix}"] / rows[f"position_impressions_{suffix}"]
    spread = rows[f"sum_position_sq_{suffix}"] / rows[f"position_impressions_{suffix}"] - rows[f"position_{suffix}"] ** 2
    rows[f"position_volatility_{suffix}"] = np.sqrt(spread.clip(lower=0))
    rows[f"ctr_{suffix}"] = clicks / impressions
    rows[f"expected_bins_{suffix}"] = client_aware_expected(rows, f"clicks_{suffix}", f"impressions_{suffix}", position_bin(rows[f"position_{suffix}"]))
    rows[f"expected_tier_{suffix}"] = client_aware_expected(rows, f"clicks_{suffix}", f"impressions_{suffix}", position_tier(rows[f"position_{suffix}"]))
    return rows


def add_luck_test(rows: pd.DataFrame, suffix: str) -> pd.DataFrame:
    """Add the negative binomial luck test and the locked shortfall rule for one month."""
    clicks, expected = rows[f"clicks_{suffix}"], rows[f"expected_bins_{suffix}"]
    rows[f"p_{suffix}"] = nbinom.cdf(clicks, 1 / ALPHA, 1 / (1 + ALPHA * expected))
    rows[f"shortfall_{suffix}"] = (rows[f"p_{suffix}"] < 0.05) & (clicks <= 0.5 * expected)
    return rows


def build_pair(month_m: str, month_next: str) -> pd.DataFrame:
    """Return one month pair's eligible rows with both months' measures and the locked target."""
    keys = ["client_hash_id", "content_hash_id"]
    first = page_months[page_months["month"] == month_m].set_index(keys)
    second = page_months[page_months["month"] == month_next].set_index(keys)
    rows = first.join(second, lsuffix="_m", rsuffix="_next", how="inner").reset_index()
    start, end = pd.Timestamp(f"{month_m}-01"), pd.Timestamp(f"{month_next}-01") + pd.offsets.MonthEnd(0)
    rows = rows[rows["client_hash_id"].isin(covered_clients(month_m))]
    rows = rows[(rows["position_impressions_m"] > 0) & (rows["position_impressions_next"] > 0)]
    rows = rows.merge(content, on="content_hash_id", how="left")
    rows = rows[~pd.to_datetime(rows["last_optimized_date"]).between(start, end)].copy()
    rows = add_month_measures(add_month_measures(rows, "m"), "next")
    rows["pair_end"] = end
    return rows.reset_index(drop=True)


def stable_pages(frame: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series]:
    """Return pages whose position moved less than one place, and each one's expected share of clicks in month M+1.

    The expected share corrects for the overall change in click rate between the two months.
    """
    stable = frame[(frame["position_next"] - frame["position_m"]).abs() < 1]
    shift = (stable["clicks_next"].sum() / stable["impressions_next"].sum()) / (stable["clicks_m"].sum() / stable["impressions_m"].sum())
    share = shift * stable["impressions_next"] / (stable["impressions_m"] + shift * stable["impressions_next"])
    return stable, share



def spread_factor(frame: pd.DataFrame) -> float:
    """Return the method-of-moments estimate of the negative binomial spread factor alpha.

    For pages with a stable position, the gap between month-M+1 clicks and the clicks their month-M rate
    predicts has variance (Poisson part) + alpha x (squared clicks). Pooling the excess over the Poisson
    part across pages gives alpha.
    """
    stable, share = stable_pages(frame)
    scale = share / (1 - share)
    gap = stable["clicks_next"] - scale * stable["clicks_m"]
    excess = (gap ** 2 - stable["clicks_next"] - scale ** 2 * stable["clicks_m"]).sum()
    squared = ((stable["clicks_next"] ** 2 - stable["clicks_next"]) + scale ** 2 * (stable["clicks_m"] ** 2 - stable["clicks_m"])).sum()
    return float(excess / squared)



pairs = {f"{month_m} → {month_next}": build_pair(month_m, month_next) for month_m, month_next in PAIRS}
ALPHA = round(float(np.median([spread_factor(rows) for rows in pairs.values()])), 3)
for rows in pairs.values():
    add_luck_test(add_luck_test(rows, "m"), "next")
    rows["target"] = rows["shortfall_next"]
    rows["missed_next"] = (rows["expected_bins_next"] - rows["clicks_next"]).clip(lower=0).where(rows["target"], 0)
show_table(f"Development pairs with the locked target (α = {ALPHA:.3f})", pd.DataFrame({
    "Rows": [f"{len(rows):,}" for rows in pairs.values()],
    "Positives": [f"{rows['target'].sum():,}" for rows in pairs.values()],
    "Base rate": [f"{rows['target'].mean():.1%}" for rows in pairs.values()],
}, index=pd.Index(list(pairs), name="Pair")))

**Method:** this cell builds the modelling notebook's 25 features (the leakage check's 23 plus rule B's two month-M signals) with the same code and checks them.


In [ ]:
# The leakage check's 23 features, plus rule B's two month-M signals.
CONTENT_TYPES = sorted(content["content_type"].dropna().unique())
INTENTS = sorted(content["main_intent"].dropna().unique())
PRODUCT_FIELDS = {"last_optimized_date", "optimization_eligible_date", "is_deleted", "is_published", "content_updated_date"}


def feature_vector(rows: pd.DataFrame) -> pd.DataFrame:
    """Return the model-ready feature matrix for one pair: month-M search features and keyword-research features only."""
    features = pd.DataFrame(index=rows.index)
    features["log_impressions_m"] = np.log1p(rows["impressions_m"])
    features["log_clicks_m"] = np.log1p(rows["clicks_m"])
    features["ctr_m"] = rows["clicks_m"] / rows["impressions_m"]
    features["position_m"] = rows["position_m"]
    features["position_volatility_m"] = rows["position_volatility_m"]
    features["days_with_impressions_m"] = rows["days_with_impressions_m"]
    features["log_expected_m"] = np.log1p(rows["expected_tier_m"])
    for field in ("search_volume", "competition", "cpc", "main_intent"):
        features[f"has_{field}"] = rows[field].notna().astype(int)
    features["log_search_volume"] = np.log1p(rows["search_volume"]).fillna(0)
    features["competition"] = rows["competition"].fillna(0)
    features["log_cpc"] = np.log1p(rows["cpc"]).fillna(0)
    features["keyword_token_count"] = rows["keyword_token_count"]
    for value in CONTENT_TYPES:
        features[f"content_type={value}"] = (rows["content_type"] == value).astype(int)
    for value in INTENTS:
        features[f"intent={value}"] = (rows["main_intent"] == value).astype(int)
    features["intent=unknown"] = rows["main_intent"].isna().astype(int)
    features["log_missed_m"] = np.log1p((rows["expected_bins_m"] - rows["clicks_m"]).clip(lower=0))
    features["short_m"] = rows["shortfall_m"].astype(int)
    return features


vectors = {name: feature_vector(rows) for name, rows in pairs.items()}
FEATURES = list(next(iter(vectors.values())).columns)
for name, features in vectors.items():
    assert list(features.columns) == FEATURES, f"{name}: columns differ from the first pair."
    assert all(pd.api.types.is_numeric_dtype(dtype) for dtype in features.dtypes), f"{name}: a feature is not numeric."
    assert not features.isna().any().any(), f"{name}: a feature still has gaps."
assert not [column for column in FEATURES if column.endswith("_next")], "A month M+1 field reached the features."
assert not set(FEATURES) & PRODUCT_FIELDS, "A product field reached the features."

LABELS = {"log_impressions_m": "Impressions (log)", "log_clicks_m": "Clicks (log)", "ctr_m": "Click-through rate",
          "position_m": "Position", "position_volatility_m": "Position spread", "days_with_impressions_m": "Days with impressions",
          "log_expected_m": "Expected clicks (log)", "has_search_volume": "Has search volume", "has_competition": "Has competition",
          "has_cpc": "Has CPC", "has_main_intent": "Has search intent", "log_search_volume": "Search volume (log)",
          "competition": "Competition", "log_cpc": "CPC (log)", "keyword_token_count": "Keyword length (words)",
          "log_missed_m": "Missed clicks (log)", "short_m": "Short by the target's rule"}


def label(feature: str) -> str:
    """Return a plain-language name for a feature."""
    return LABELS.get(feature) or feature.replace("content_type=", "Content type: ").replace("intent=", "Search intent: ")
show_table("Feature vector, every pair", pd.DataFrame({
    "Rows": [f"{len(features):,}" for features in vectors.values()],
    "Features": [features.shape[1] for features in vectors.values()],
    "Gaps left": [int(features.isna().sum().sum()) for features in vectors.values()],
}, index=pd.Index(list(vectors), name="Pair")))

## 1. Two paper findings + my methodology questions

The FlyRank paper, _The State of AI-Driven SEO_ (March 2026), studies 341,701 pages across 57 brands. Two of its findings bear directly on click-through, the subject of this project. For each, the three questions are the same: where does the number come from, what does the design actually test, and would the claim survive a stricter test? The aim is the next level of rigor, not a scalp; the paper itself flags several of these limits.


### 1a. Finding #3: "Click capture by position tier"

**The claim.** Weighted CTR falls sharply as pages move down the results: 0.423% in the top 3, 0.339% on the rest of page 1, 0.325% at positions 11 to 20, 0.163% at 21 to 50, and 0.050% beyond, an 88% drop. The paper concludes that refining pages already on page 1 is a more reliable source of extra clicks.

**Where the number comes from.** Total clicks ÷ total impressions inside each of five position tiers, over a rolling 90-day window. It is a descriptive aggregate, with no model and no test set.

**My methodology questions.**

1. _Does a tier average hide the shape inside the tier?_ A page's average position blends many searches, and the earlier signal audit found the curve peaking at positions 2 and 3 and dipping near 7, so the top-3 and page-1 averages may each hide a non-monotonic curve.
2. _Is the drop a position effect or a site mix?_ If large sites with low click rates dominate the lower tiers, part of the drop belongs to the sites, not the positions. A per-client spread inside each tier shows how much sites differ.
3. _Does the 90-day position overlap the 90-day clicks?_ Position and clicks come from the same window, so the comparison is fine as a description but cannot say what moving a page up would do.

**Method:** this cell recomputes weighted CTR on every page-month with a valid position, February to May 2026, with the paper's five tiers (using each page's impression-weighted monthly position), then by single position from 1 to 20, and then client by client inside each tier, among clients with at least 10,000 impressions in that tier.


In [ ]:
# The paper's five tiers on the warehouse, then single positions, then the spread across clients.
PAPER_CTR = {"Top 3": 0.00423, "Page 1 (4 to 10)": 0.00339, "Striking (11 to 20)": 0.00325, "Page 3 to 5 (21 to 50)": 0.00163, "Deep (50+)": 0.00050}
visible = page_months[page_months["position_impressions"] > 0].copy()
visible["position"] = visible["sum_position"] / visible["position_impressions"]
visible["tier"] = pd.cut(visible["position"], [0, 3.5, 10.5, 20.5, 50.5, np.inf], labels=list(PAPER_CTR))
tiers = visible.groupby("tier", observed=True)[["clicks", "impressions"]].sum()
tiers["ctr"] = tiers["clicks"] / tiers["impressions"]
show_table("Weighted CTR by the paper's position tiers: paper against warehouse (February to May 2026)", pd.DataFrame({
    "Paper": [f"{PAPER_CTR[tier]:.3%}" for tier in tiers.index],
    "Warehouse": tiers["ctr"].map("{:.3%}".format),
    "Page-months": visible["tier"].value_counts().reindex(tiers.index).map("{:,}".format),
    "Impressions": tiers["impressions"].map("{:,.0f}".format),
}).rename_axis("Tier"))

near = visible[visible["position"] < 20.5].assign(rank=lambda frame: frame["position"].round().clip(lower=1).astype(int))
curve = near.groupby("rank")[["clicks", "impressions"]].sum()
curve["ctr"] = curve["clicks"] / curve["impressions"]
rises = int((curve["ctr"].diff() > 0).sum())
show_table("Weighted CTR by single position, 1 to 20", pd.DataFrame({
    "Weighted CTR": curve["ctr"].map("{:.3%}".format),
    "Change from the position above": curve["ctr"].diff().map(lambda change: "" if pd.isna(change) else f"{change:+.3%}"),
    "Impressions": curve["impressions"].map("{:,.0f}".format),
}).rename_axis("Position"))

per_client = visible.groupby(["tier", "client_hash_id"], observed=True)[["clicks", "impressions"]].sum()
per_client = per_client[per_client["impressions"] >= 10_000]
per_client["ctr"] = per_client["clicks"] / per_client["impressions"]
spread = per_client.groupby(level="tier", observed=True)["ctr"].describe(percentiles=[0.1, 0.5, 0.9])
show_table("Client-level weighted CTR inside each tier (clients with 10,000+ impressions in the tier)", pd.DataFrame({
    "Clients": spread["count"].map("{:.0f}".format),
    "10th percentile": spread["10%"].map("{:.3%}".format),
    "Median": spread["50%"].map("{:.3%}".format),
    "90th percentile": spread["90%"].map("{:.3%}".format),
    "90th ÷ 10th": (spread["90%"] / spread["10%"]).map("{:.1f}x".format),
}).rename_axis("Tier"))
show("Shape of the curve", {"positions 2 to 20 where CTR rises above the position before": f"{rises} of {len(curve) - 1}",
                            "warehouse drop from top 3 to deep": f"{1 - tiers['ctr'].iloc[-1] / tiers['ctr'].iloc[0]:.0%} (paper: 88%)"})

### 1b. ML appendix: "What predicts health?"

**The claim.** A random forest predicts the health score, and position (43%), impressions (32%), scroll depth (15%), and CTR (8%) are its most important features. The model is described as holdout-tested.

**Where the label comes from.** The health score is a FlyRank composite: impressions (30 points) + position (30) + CTR (20) + scroll depth (20). The four top features are the four ingredients of the target, in nearly the order of their weights. The paper says so itself: "health score is partly constructed from inputs such as position and impressions. High importance is therefore expected."

**What the design tests.** An 80/20 random split of 61,790 active pages. Pages of the same brand sit on both sides, and features and target share one window.

**My methodology questions.**

1. _What does a high holdout score mean when the target is a formula of the inputs?_ It shows the model can rebuild the formula, not that it found anything about pages.
2. _Would the score survive a client-grouped split and a future target?_ If the model is asked for next month's health on brands it never saw, the formula can no longer be copied.
3. _How would the importance ranking change?_ With a future target, the ranking would show what predicts health, rather than how health is computed.

**Method:** this cell demonstrates the first two questions on the warehouse. It rebuilds an approximate health score from the paper's formula (impressions, position, and CTR as within-month percentile ranks, weighted 30, 30, and 20; scroll depth is not in the warehouse, so the score runs to 80), then trains the same kind of random forest three ways on the Mar→Apr pair: the paper's design (same-month score, random rows), the same score with clients grouped, and next month's score with clients grouped. It reports R² on the held-out fifth and the forest's built-in importances under the paper's design.


In [ ]:
SEED = 42


def health_like(rows: pd.DataFrame, suffix: str) -> pd.Series:
    """Return an approximate health score from the paper's formula, without scroll depth (maximum 80)."""
    impressions = rows[f"impressions_{suffix}"].rank(pct=True)
    position = 1 - rows[f"position_{suffix}"].rank(pct=True)
    ctr = (rows[f"clicks_{suffix}"] / rows[f"impressions_{suffix}"]).rank(pct=True)
    return 30 * impressions + 30 * position + 20 * ctr


def forest() -> RandomForestRegressor:
    """Return a random forest regressor with fixed settings."""
    return RandomForestRegressor(n_estimators=100, min_samples_leaf=50, max_samples=0.25, n_jobs=-1, random_state=SEED)


names = list(pairs)
demo_name = names[1]
demo_rows, demo_features = pairs[demo_name], vectors[demo_name]
random_fold = next(KFold(n_splits=5, shuffle=True, random_state=SEED).split(demo_features))
client_fold = next(GroupKFold(n_splits=5).split(demo_features, groups=demo_rows["client_hash_id"]))
designs = {"Paper's design: same-month score, random rows": (health_like(demo_rows, "m"), random_fold),
           "Same-month score, clients grouped": (health_like(demo_rows, "m"), client_fold),
           "Next month's score, clients grouped": (health_like(demo_rows, "next"), client_fold)}
fits = {}
for design, (score, (train, test)) in designs.items():
    model = forest().fit(demo_features.iloc[train], score.iloc[train])
    fits[design] = (r2_score(score.iloc[test], model.predict(demo_features.iloc[test])), model)
show_table(f"Predicting an approximate health score, {demo_name} (R² on the held-out fifth)", pd.DataFrame({
    "R²": [f"{r2:.3f}" for r2, _ in fits.values()]}, index=pd.Index(list(fits), name="Design")))
paper_model = fits["Paper's design: same-month score, random rows"][1]
built_in = pd.Series(paper_model.feature_importances_, index=FEATURES).sort_values(ascending=False).head(6)
show_table("Built-in importances under the paper's design (top six)", pd.DataFrame({
    "Importance": built_in.map("{:.0%}".format)}).rename(index=label).rename_axis("Feature"))

## 2. My model under an honest split (before/after)

This section tests the modelling notebook's result four ways: under looser and stricter designs (2a), on the sealed month nobody has looked at (2b), with an interval on the gap (2c), and segment by segment (2d).


### 2a. The same model under four designs

The modelling notebook judged logistic regression with the strictest design: trained on earlier months, tested on later ones, on clients it never saw. Here the same model is trained four ways, from the loosest design to the strictest, and every version is scored on exactly the same test pages, so the differences show how much each protection matters.

| Design                                                      | Training sees                                                         | What it would flatter                              |
| ----------------------------------------------------------- | --------------------------------------------------------------------- | -------------------------------------------------- |
| **Random rows** (the paper's design)                        | Other pages of the same sites, in the same months and the test months | Memorized sites and future months                  |
| **Client-grouped, same months**                             | Other sites, including in the test month itself                       | Future months                                      |
| **Time-forward, same clients**                              | The same sites, in earlier months only                                | Memorized sites                                    |
| **Walk-forward + client** (the modelling notebook's design) | Other sites, in earlier months only                                   | Nothing: it is the real-use question on a new site |

**Method:** this cell scores both test pairs (Mar→Apr and Apr→May) with logistic regression under the four designs, with rule B beside them, and reports Precision@50, Capture@50, and ROC-AUC for each, then the means and how far each design sits above the strictest one. The random and client-grouped designs use five folds over all three development pairs pooled; the time-forward design trains one model per test pair on every earlier pair.


In [ ]:
BAND = (1_000, 9_999)


def logistic() -> Pipeline:
    """Return logistic regression with the modelling notebook's fixed settings."""
    return make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=2_000))


def rule_b(rows: pd.DataFrame) -> pd.Series:
    """The frozen baseline: missed clicks against the position-bin expectation, kept where the page fell short in month M."""
    return (rows["expected_bins_m"] - rows["clicks_m"]).clip(lower=0).where(rows["shortfall_m"], 0)


def top_k_total(score: pd.Series, values: pd.Series, k: int) -> float:
    """Return the expected total of `values` over the top k rows, counting rows tied at the cut-off by their average."""
    k = min(k, len(score))
    cut = score.sort_values(ascending=False).iloc[k - 1]
    above = values[score > cut]
    return float(above.sum() + (k - len(above)) * values[score == cut].mean())


def metric_row(rows: pd.DataFrame, score: pd.Series) -> dict[str, float]:
    """Return the comparison metrics for one score on one set of pages."""
    band = rows["impressions_m"].between(*BAND)
    return {"Precision@50": top_k_total(score, rows["target"].astype(float), 50) / 50,
            "Precision@20": top_k_total(score, rows["target"].astype(float), 20) / 20,
            "Capture@50": top_k_total(score, rows["missed_next"], 50) / float(rows["missed_next"].sum()),
            "Precision@50, 1,000 to 9,999 impressions": top_k_total(score[band], rows.loc[band, "target"].astype(float), 50) / 50,
            "ROC-AUC": roc_auc_score(rows["target"], score),
            "Average precision": average_precision_score(rows["target"], score),
            "Base rate": rows["target"].mean()}


TESTS = {names[i]: names[:i] for i in range(1, len(names))}
all_rows = pd.concat([pairs[name].assign(pair=name) for name in names], ignore_index=True)
all_features = pd.concat([vectors[name] for name in names], ignore_index=True)
all_target = all_rows["target"].astype(int)


def pooled_out_of_fold(splits: list) -> pd.Series:
    """Score every development row with a model trained on the other folds of the pooled pairs."""
    score = pd.Series(np.nan, index=all_rows.index)
    for train, test in splits:
        score.iloc[test] = logistic().fit(all_features.iloc[train], all_target.iloc[train]).predict_proba(all_features.iloc[test])[:, 1]
    return score


random_score = pooled_out_of_fold(list(KFold(n_splits=5, shuffle=True, random_state=SEED).split(all_features)))
grouped_score = pooled_out_of_fold(list(GroupKFold(n_splits=5).split(all_features, groups=all_rows["client_hash_id"])))
DESIGNS = ["Random rows", "Client-grouped, same months", "Time-forward, same clients", "Walk-forward + client"]
split_scores = {design: {} for design in DESIGNS}
for test, trained_on in TESTS.items():
    test_rows, test_features = pairs[test], vectors[test]
    in_test = (all_rows["pair"] == test).to_numpy()
    split_scores["Random rows"][test] = pd.Series(random_score[in_test].to_numpy(), index=test_rows.index)
    split_scores["Client-grouped, same months"][test] = pd.Series(grouped_score[in_test].to_numpy(), index=test_rows.index)
    train_rows = pd.concat([pairs[name] for name in trained_on], ignore_index=True)
    train_features = pd.concat([vectors[name] for name in trained_on], ignore_index=True)
    train_target = train_rows["target"].astype(int)
    split_scores["Time-forward, same clients"][test] = pd.Series(
        logistic().fit(train_features, train_target).predict_proba(test_features)[:, 1], index=test_rows.index)
    walk = pd.Series(np.nan, index=test_rows.index)
    for _, test_index in GroupKFold(n_splits=5).split(test_rows, groups=test_rows["client_hash_id"]):
        keep = ~train_rows["client_hash_id"].isin(set(test_rows.loc[test_index, "client_hash_id"]))
        walk.iloc[test_index] = logistic().fit(train_features[keep], train_target[keep]).predict_proba(test_features.iloc[test_index])[:, 1]
    split_scores["Walk-forward + client"][test] = walk

split_results = pd.DataFrame(
    [{"Test pair": test, "Design": design, **metric_row(pairs[test], split_scores[design][test])} for test in TESTS for design in DESIGNS]
    + [{"Test pair": test, "Design": "Rule B (no training)", **metric_row(pairs[test], rule_b(pairs[test]))} for test in TESTS])
show_table("Logistic regression under four designs, same test pages", split_results.sort_values(["Test pair"], kind="stable").assign(
    **{column: split_results[column].map("{:.2f}".format) for column in ["Precision@50", "Precision@20"]},
    **{"Capture@50": split_results["Capture@50"].map("{:.1%}".format), "ROC-AUC": split_results["ROC-AUC"].map("{:.3f}".format)},
)[["Test pair", "Design", "Precision@50", "Precision@20", "Capture@50", "ROC-AUC"]].set_index(["Test pair", "Design"]))
split_means = split_results.groupby("Design")[["Precision@50", "Capture@50", "ROC-AUC"]].mean().loc[[*DESIGNS, "Rule B (no training)"]]
strict = split_means.loc["Walk-forward + client"]
show_table("Means over the two test pairs, and how far each design sits above the strictest", pd.DataFrame({
    "Mean Precision@50": split_means["Precision@50"].map("{:.3f}".format),
    "Above the strictest design": (split_means["Precision@50"] - strict["Precision@50"]).map("{:+.3f}".format),
    "Mean ROC-AUC": split_means["ROC-AUC"].map("{:.3f}".format),
    "ROC-AUC above the strictest": (split_means["ROC-AUC"] - strict["ROC-AUC"]).map("{:+.3f}".format),
    "Mean Capture@50": split_means["Capture@50"].map("{:.2%}".format),
}).rename_axis("Design"))

### 2b. The sealed May→June test

The May→June pair has not been read by any notebook in this project. It is scored here once, with everything fixed in advance: the target (with the spread factor estimated on the development pairs, 0.039), the 25 features, logistic regression's settings, and the comparison rule from the baseline and modelling notebooks (more than 0.02 ahead on Precision@50, or within 0.02 and ahead on Capture@50). The models are trained on all three development pairs.

- **Primary (unseen clients):** the sealed pair's clients are split into five folds, and each fold is scored by a model that never saw those clients, exactly the claim the modelling notebook made.
- **Context (deployment):** one model trained on every client scores every page, which is how the queue would run for existing clients.

**Method:** this cell reads June 2026 for the first time, builds the sealed pair with the data contract's rules, scores it with rule B and both logistic regression designs, and applies the comparison rule to the primary design.


In [ ]:
SEALED_PAIR = ("2026-05", "2026-06")
assert SEALED_PAIR[1] not in DEV_MONTHS, "The sealed month must never be a development month."
page_months = pd.concat([page_months, cached(f"page_month_v3_{SEALED_PAIR[1]}", PAGE_MONTH_SQL.format(month=SEALED_PAIR[1], source=month_files([SEALED_PAIR[1]])))], ignore_index=True)
sealed_name = f"{SEALED_PAIR[0]} → {SEALED_PAIR[1]} (sealed)"
sealed = build_pair(*SEALED_PAIR)
add_luck_test(add_luck_test(sealed, "m"), "next")  # the spread factor stays the one estimated on the development pairs
sealed["target"] = sealed["shortfall_next"]
sealed["missed_next"] = (sealed["expected_bins_next"] - sealed["clicks_next"]).clip(lower=0).where(sealed["target"], 0)
sealed_features = feature_vector(sealed)
assert list(sealed_features.columns) == FEATURES and not sealed_features.isna().any().any(), "The sealed features differ from the development features."

primary = pd.Series(np.nan, index=sealed.index)
for _, test_index in GroupKFold(n_splits=5).split(sealed, groups=sealed["client_hash_id"]):
    keep = ~all_rows["client_hash_id"].isin(set(sealed.loc[test_index, "client_hash_id"]))
    primary.iloc[test_index] = logistic().fit(all_features[keep], all_target[keep]).predict_proba(sealed_features.iloc[test_index])[:, 1]
sealed_scores = {"Rule B (baseline)": rule_b(sealed), "Logistic regression, unseen clients (primary)": primary,
                 "Logistic regression, all clients (deployment)": pd.Series(
                     logistic().fit(all_features, all_target).predict_proba(sealed_features)[:, 1], index=sealed.index)}
sealed_results = pd.DataFrame({name: metric_row(sealed, score) for name, score in sealed_scores.items()}).T
show_table(f"The sealed pair: {len(sealed):,} pages, {int(sealed['target'].sum()):,} real shortfalls, α = {ALPHA:.3f}", sealed_results.assign(
    **{column: sealed_results[column].map("{:.2f}".format) for column in ["Precision@50", "Precision@20", "Precision@50, 1,000 to 9,999 impressions"]},
    **{column: sealed_results[column].map("{:.3f}".format) for column in ["ROC-AUC", "Average precision"]},
    **{"Capture@50": sealed_results["Capture@50"].map("{:.1%}".format), "Base rate": sealed_results["Base rate"].map("{:.1%}".format)},
).rename_axis("Scorer"))
model_row, base_row = sealed_results.loc["Logistic regression, unseen clients (primary)"], sealed_results.loc["Rule B (baseline)"]
gap = round(model_row["Precision@50"] - base_row["Precision@50"], 6)
sealed_verdict = ("beats rule B" if gap > 0.02 or (abs(gap) <= 0.02 and model_row["Capture@50"] > base_row["Capture@50"])
                  else "does not beat rule B")
show("Sealed verdict, by the rule fixed in advance", {"logistic regression (unseen clients)": sealed_verdict,
                                                      "Precision@50 gap": f"{gap:+.2f}",
                                                      "Capture@50": f"{model_row['Capture@50']:.1%} against {base_row['Capture@50']:.1%}"})

### 2c. How certain is the gap?

Two test pairs and a sealed pair give three gaps, each built on 50 pages. A gap of a few pages could come from which sites happened to be in the data. The client-level bootstrap asks how much the gap would move with a different mix of sites: whole clients are resampled with replacement, both top 50s are rebuilt on the resampled pages, and the gap is recorded, 1,000 times. Clients rather than pages are resampled because a site's pages rise and fall together; resampling single pages would make the interval look falsely narrow.

**Method:** this cell runs the bootstrap on each of the three test sets (the two development test pairs under the strictest design and the sealed pair's primary design) and reports the observed gaps in Precision@50 and Capture@50, their 95% intervals, and the share of resamples in which logistic regression comes out ahead.


In [ ]:
REPS = 1_000
test_sets = {test: (pairs[test], split_scores["Walk-forward + client"][test]) for test in TESTS}
test_sets[sealed_name] = (sealed, primary)


def bootstrap_gaps(rows: pd.DataFrame, model_score: pd.Series, k: int = 50) -> np.ndarray:
    """Return REPS client-level bootstrap draws of the Precision@k and Capture@k gaps (logistic regression minus rule B)."""
    rng = np.random.default_rng(SEED)
    members = list(rows.groupby("client_hash_id").indices.values())
    model, base = model_score.to_numpy(), rule_b(rows).to_numpy()
    target, value = rows["target"].to_numpy(dtype=float), rows["missed_next"].to_numpy()
    draws = np.empty((REPS, 2))
    for rep in range(REPS):
        index = np.concatenate([members[i] for i in rng.integers(len(members), size=len(members))])
        top_model, top_base = (index[np.argpartition(-score[index], k)[:k]] for score in (model, base))
        total = value[index].sum()
        draws[rep] = (target[top_model].mean() - target[top_base].mean(), (value[top_model].sum() - value[top_base].sum()) / total)
    return draws


bootstrap = {}
for name, (rows, score) in test_sets.items():
    draws = bootstrap_gaps(rows, score)
    observed, base = metric_row(rows, score), metric_row(rows, rule_b(rows))
    bootstrap[name] = {"Precision@50 gap": observed["Precision@50"] - base["Precision@50"],
                       "Precision low": np.percentile(draws[:, 0], 2.5), "Precision high": np.percentile(draws[:, 0], 97.5),
                       "Model ahead": (draws[:, 0] > 0).mean(),
                       "Capture@50 gap": observed["Capture@50"] - base["Capture@50"],
                       "Capture low": np.percentile(draws[:, 1], 2.5), "Capture high": np.percentile(draws[:, 1], 97.5)}
bootstrap = pd.DataFrame(bootstrap).T
show_table(f"Logistic regression minus rule B, client-level bootstrap ({REPS:,} resamples, 95% intervals)", pd.DataFrame({
    "Precision@50 gap": bootstrap["Precision@50 gap"].map("{:+.2f}".format),
    "95% interval": [f"{low:+.2f} to {high:+.2f}" for low, high in zip(bootstrap["Precision low"], bootstrap["Precision high"], strict=True)],
    "Resamples with the model ahead": bootstrap["Model ahead"].map("{:.0%}".format),
    "Capture@50 gap (points)": (100 * bootstrap["Capture@50 gap"]).map("{:+.1f}".format),
    "Capture 95% interval (points)": [f"{100 * low:+.1f} to {100 * high:+.1f}" for low, high in zip(bootstrap["Capture low"], bootstrap["Capture high"], strict=True)],
}).rename_axis("Test set"))

**Method:** this cell draws the three Precision@50 gaps with their 95% intervals: a dot for the observed gap, a line for the interval, and a dashed line at zero, where the model and rule B are level. The title states the finding; the table above holds the values.


In [ ]:
def load_inter() -> str:
    """Register the Inter font for this session and return the font family to draw with.

    The 400 and 600 weights are cached in the system temp folder, never in the project. If the
    download is unavailable, the chart falls back to matplotlib's default font, so the notebook still runs offline.
    """
    cache = os.path.join(tempfile.gettempdir(), "inter-font")
    os.makedirs(cache, exist_ok=True)
    try:
        for weight in (400, 600):
            path = os.path.join(cache, f"inter-{weight}.ttf")
            if not os.path.exists(path):
                urllib.request.urlretrieve(f"https://cdn.jsdelivr.net/fontsource/fonts/inter@latest/latin-{weight}-normal.ttf", path)
            font_manager.fontManager.addfont(path)
    except OSError:
        return "DejaVu Sans"
    return "Inter"


INK, SECONDARY, MUTED, GRID, ZERO, SURFACE, SERIES = "#0b0b0b", "#52514e", "#8a8984", "#ecebe7", "#9a9893", "#fcfcfb", "#2a78d6"
plt.rcParams.update({"font.family": load_inter()})
rows_drawn = bootstrap.iloc[::-1]
fig, ax = plt.subplots(figsize=(10, 4.6), dpi=120, facecolor=SURFACE)
fig.subplots_adjust(left=0.27, right=0.95, top=0.72, bottom=0.16)
ax.set_facecolor(SURFACE)
ax.axvline(0, color=ZERO, linewidth=1.3, linestyle=(0, (4, 3)), zorder=1)
for row, (_, values) in enumerate(rows_drawn.iterrows()):
    ax.plot([values["Precision low"], values["Precision high"]], [row, row], color=SERIES, linewidth=3, solid_capstyle="round", zorder=2)
    ax.plot(values["Precision@50 gap"], row, "o", markersize=11, color=SERIES, markeredgecolor=SURFACE, markeredgewidth=2, zorder=3)
    ax.annotate(f"{values['Precision@50 gap']:+.2f}", (values["Precision high"], row), xytext=(10, 0), textcoords="offset points",
                va="center", color=SECONDARY, fontsize=10.5, fontweight="semibold")


def month_words(name: str) -> str:
    """Write a pair such as '2026-04 → 2026-05' as 'April to May 2026'; the Inter font has no arrow glyph."""
    first, second = (pd.Timestamp(f"{month}-01") for month in re.findall(r"\d{4}-\d{2}", name))
    return f"{first:%B} to {second:%B %Y}" + (" (sealed)" if "sealed" in name else "")


ax.set_yticks(range(len(rows_drawn)), [month_words(name) for name in rows_drawn.index])
reach = max(abs(rows_drawn["Precision low"]).max(), abs(rows_drawn["Precision high"]).max()) * 1.35
ax.set_xlim(-reach, reach)
ax.set_ylim(-0.6, len(rows_drawn) - 0.4)
ax.xaxis.set_major_formatter(mpl.ticker.FuncFormatter(lambda value, _: f"{value:+.2f}" if value else "0"))
ax.set_xlabel("Precision@50, logistic regression minus rule B (positive = the model hits more real shortfalls)", color=MUTED, fontsize=10, labelpad=8)
ax.grid(axis="x", color=GRID, linewidth=1)
ax.set_axisbelow(True)
ax.tick_params(axis="x", length=0, colors=MUTED, labelsize=9.5, pad=6)
ax.tick_params(axis="y", length=0, colors=INK, labelsize=10.5, pad=8)
for spine in ax.spines.values():
    spine.set_visible(False)
above_zero, ahead = int((bootstrap["Precision low"] > 0).sum()), int((bootstrap["Precision@50 gap"] > 0).sum())
if above_zero == len(bootstrap):
    title = f"Logistic regression's lead over rule B holds in all {len(bootstrap)} test sets"
elif above_zero == 0 and ahead == len(bootstrap):
    title = "Logistic regression leads rule B in every test set, but no single lead is clear of chance"
else:
    title = f"Logistic regression leads rule B in {ahead} of {len(bootstrap)} test sets, clearly in {above_zero}"
fig.text(0.5, 0.92, title, ha="center", color=INK, fontsize=16, fontweight="semibold")
fig.text(0.5, 0.855, f"Observed gap (dot) and 95% interval from {REPS:,} client-level resamples (line). An interval clear of zero is a lead a different mix of sites would not erase.",
         ha="center", color=SECONDARY, fontsize=9.5)
plt.show()

### 2d. Does it hold in every segment?

An average can hide a model that wins big in one corner and loses everywhere else. The segments are the two axes the earlier notebooks found matter most (page size and search intent), the content type, and the client.

**Method:** this cell reports Precision@50 for rule B and logistic regression inside each size band, content type, and search intent, on each of the three test sets, with page counts and base rates; segments with fewer than 500 pages or 50 real shortfalls are marked as small. It then compares the two scorers client by client: Precision@10 inside each client with at least 10 real shortfalls, counting the clients where the model does better, the same, or worse.


In [ ]:
SMALL_PAGES, SMALL_POSITIVES, CLIENT_MIN_POSITIVES = 500, 50, 10
SIZE_BANDS = {"Under 100": (0, 99), "100 to 999": (100, 999), "1,000 to 9,999": (1_000, 9_999), "10,000 or more": (10_000, np.inf)}


def segment_masks(rows: pd.DataFrame) -> dict[tuple[str, str], pd.Series]:
    """Return a True/False mask for every segment of one test set."""
    masks = {("Size", band): rows["impressions_m"].between(low, high) for band, (low, high) in SIZE_BANDS.items()}
    masks |= {("Content type", value): rows["content_type"] == value for value in sorted(rows["content_type"].dropna().unique())}
    intent = rows["main_intent"].fillna("unknown")
    masks |= {("Search intent", value): intent == value for value in sorted(intent.unique())}
    return masks


segment_rows = []
for name, (rows, score) in test_sets.items():
    base = rule_b(rows)
    for (kind, value), inside in segment_masks(rows).items():
        pages, positives = int(inside.sum()), int(rows.loc[inside, "target"].sum())
        if pages == 0:
            continue
        precision = {scorer: top_k_total(series[inside], rows.loc[inside, "target"].astype(float), 50) / min(50, pages)
                     for scorer, series in (("Rule B", base), ("Logistic regression", score))}
        lead = precision["Logistic regression"] - precision["Rule B"]
        segment_rows.append({"Segment": kind, "Group": value, "Test set": name, "Pages": f"{pages:,}",
                             "Base rate": f"{positives / pages:.1%}", "Rule B": f"{precision['Rule B']:.2f}",
                             "Logistic regression": f"{precision['Logistic regression']:.2f}",
                             "Model ahead": "yes" if lead > 0 else ("level" if lead == 0 else "no"),
                             "Small segment": "small" if pages < SMALL_PAGES or positives < SMALL_POSITIVES else ""})
show_table("Precision@50 inside each segment", pd.DataFrame(segment_rows).set_index(["Segment", "Group", "Test set"]))

client_rows = []
for name, (rows, score) in test_sets.items():
    base, outcome = rule_b(rows), []
    for _, members in rows.groupby("client_hash_id"):
        if members["target"].sum() < CLIENT_MIN_POSITIVES:
            continue
        target = members["target"].astype(float)
        outcome.append(np.sign(top_k_total(score[members.index], target, 10) - top_k_total(base[members.index], target, 10)))
    outcome = pd.Series(outcome, dtype=float)
    client_rows.append({"Test set": name, "Clients compared": len(outcome), "Model better": int((outcome > 0).sum()),
                        "Level": int((outcome == 0).sum()), "Rule B better": int((outcome < 0).sum())})
show_table(f"Client by client: Precision@10 inside each client with {CLIENT_MIN_POSITIVES}+ real shortfalls", pd.DataFrame(client_rows).set_index("Test set"))

## 3. Leakage audit

The leakage check of week 3 audited 23 features. The final model uses 25: the same 23 plus two signals from rule B, which are new and have never been audited. The guide's five questions are answered in order, each with a test in code rather than a sentence alone.

1. **Is every feature available before the decision point?** Tested by scrambling every month M+1 column, the target, and every product field, then rebuilding the features: if any feature changes, it read something it should not.
2. **Does the feature window overlap the target window?** Month M and month M+1 are distinct calendar months by construction; the time-travel test shows how much the two new features would separate the target if they were built from month M+1 instead.
3. **Did a rebuilt product flag sneak in?** No FlyRank product field is a feature, and the scramble test covers them too.
4. **Does a derived field secretly encode the target?** The two new features apply the target's own formula to month M. The separation scan checks every feature against the 0.90 alarm used in week 3, and the planted leak (month M+1's clicks ÷ expected) shows the harness still catches a real leak.
5. **Are related rows split across train and test in a way that makes the test too easy?** Pages repeat across month pairs, so the audit counts test pages that also appear in their model's training rows, under the strictest design and under the time-forward design.

**Method:** this cell runs the five checks on the development pairs and the sealed pair and prints the result of each.


In [ ]:
rng = np.random.default_rng(SEED)

# 1 and 3. Scramble every month M+1 column, the target, and every product field; the features must not change.
for name, rows in {**pairs, sealed_name: sealed}.items():
    scrambled = rows.copy()
    for column in [column for column in rows.columns if column.endswith("_next") or column in PRODUCT_FIELDS | {"target", "missed_next"}]:
        scrambled[column] = rng.permutation(scrambled[column].to_numpy())
    assert feature_vector(scrambled).equals(feature_vector(rows)), f"{name}: a feature reads month M+1, the target, or a product field."
assert not set(FEATURES) & PRODUCT_FIELDS, "A product field is a feature."


def separation(values: pd.Series, target: pd.Series) -> float:
    """Return max(AUC, 1 - AUC) of one column against the target."""
    auc = roc_auc_score(target, values)
    return max(auc, 1 - auc)


def with_planted_leak(rows: pd.DataFrame, features: pd.DataFrame) -> pd.DataFrame:
    """Return the features plus month M+1's clicks ÷ expected clicks, the target's own ingredient."""
    return features.assign(planted_leak=rows["clicks_next"] / rows["expected_bins_next"])


# 2. Time travel for the two new features: built from month M (as used) against built from month M+1.
travel = []
for name, rows in pairs.items():
    missed_next = np.log1p((rows["expected_bins_next"] - rows["clicks_next"]).clip(lower=0))
    travel.append({"Pair": name, "Missed clicks, month M": separation(vectors[name]["log_missed_m"], rows["target"]),
                   "Missed clicks, month M+1": separation(missed_next, rows["target"]),
                   "Short by the rule, month M": separation(vectors[name]["short_m"], rows["target"]),
                   "Short by the rule, month M+1": separation(rows["shortfall_next"].astype(int), rows["target"])})
show_table("Time travel: separation of the two new features if built from the wrong month", pd.DataFrame(travel).set_index("Pair").map("{:.2f}".format))

# 4. Separation scan of all 25 features with the 0.90 alarm; alarms re-checked among pages with 1,000+ impressions.
scan = pd.DataFrame({name: {feature: separation(vectors[name][feature], rows["target"]) for feature in FEATURES} for name, rows in pairs.items()})
alarms = list(scan.index[(scan >= 0.90).any(axis=1)])
show_table("Separation of every feature (0.90 = alarm)", scan.map("{:.2f}".format).assign(
    Alarm=["yes" if feature in alarms else "" for feature in scan.index]).rename(index=label).rename_axis("Feature"))
if alarms:
    show_table("Alarmed features among pages with 1,000+ impressions", pd.DataFrame({
        name: {label(feature): f"{separation(vectors[name].loc[rows['impressions_m'] >= 1_000, feature], rows.loc[rows['impressions_m'] >= 1_000, 'target']):.2f}"
               for feature in alarms} for name, rows in pairs.items()}).rename_axis("Feature"))

last_test = names[-1]
train_rows = pd.concat([pairs[name] for name in TESTS[last_test]], ignore_index=True)
train_features = pd.concat([vectors[name] for name in TESTS[last_test]], ignore_index=True)
test_rows, test_features = pairs[last_test], vectors[last_test]
honest_fit = logistic().fit(train_features, train_rows["target"].astype(int))
leaky_fit = logistic().fit(with_planted_leak(train_rows, train_features), train_rows["target"].astype(int))
planted = {"Honest features": metric_row(test_rows, pd.Series(honest_fit.predict_proba(test_features)[:, 1], index=test_rows.index)),
           "Plus the planted leak": metric_row(test_rows, pd.Series(leaky_fit.predict_proba(with_planted_leak(test_rows, test_features))[:, 1], index=test_rows.index))}
show_table(f"Planted leak, time-forward design, {last_test}", pd.DataFrame(planted).T[["Precision@50", "ROC-AUC", "Average precision"]].map("{:.3f}".format).rename_axis("Features"))

# 5. Test pages that also appear in their model's training rows (by page id).
overlap = []
for name, (rows, _) in test_sets.items():
    training = all_rows if name == sealed_name else pd.concat([pairs[earlier] for earlier in TESTS[name]], ignore_index=True)
    strict_count = 0
    for _, test_index in GroupKFold(n_splits=5).split(rows, groups=rows["client_hash_id"]):
        held = set(rows.loc[test_index, "client_hash_id"])
        kept_pages = set(training.loc[~training["client_hash_id"].isin(held), "content_hash_id"])
        strict_count += int(rows.loc[test_index, "content_hash_id"].isin(kept_pages).sum())
    assert strict_count == 0, f"{name}: a test page reached its model's training rows."
    overlap.append({"Test set": name, "Test pages": f"{len(rows):,}", "Seen in training, strictest design": strict_count,
                    "Seen in training, time-forward design": f"{int(rows['content_hash_id'].isin(set(training['content_hash_id'])).sum()):,}"})
show_table("Test pages that also appear in their model's training rows", pd.DataFrame(overlap).set_index("Test set"))
show("Leakage audit (all checks passed)", {
    "1. features available before the decision point": "yes: scrambling every month M+1 column and the target leaves all 25 features unchanged",
    "2. feature and target windows": "disjoint calendar months; the month M+1 versions of the new features are shown above for contrast",
    "3. product flags": "none is a feature; scrambling them leaves the features unchanged",
    "4. derived fields encoding the target": f"{len(alarms)} feature(s) at or above the 0.90 alarm; the planted leak is caught",
    "5. related rows across train and test": "0 test pages in their model's training rows under the strictest design",
})

## 4. Claim rewrite

Three of the boldest sentences written in the earlier notebooks, one for each rung of the claim ladder, are put through the three reviewer questions: where does the label come from, what does the validation design actually test, and would the number survive a stricter split? The rewritten sentences follow the evidence printed below.

| #   | Original sentence                                                                                     | Notebook      | What the evidence is                                                                                                                                          |
| --- | ----------------------------------------------------------------------------------------------------- | ------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| 1   | "**Logistic regression beats rule B and is carried forward.**"                                        | Modelling     | A validated ranking on unseen clients and months: it may say "ranks" and "flags", with the precision, the interval, and the sealed result beside it           |
| 2   | "A monthly review queue built on real shortfalls is pointing at lasting problems, not passing noise." | Signal audit  | A measured association between this month's shortfall and next month's: it may say "is associated with", not "will continue" or "is worth fixing"             |
| 3   | "FlyRank's team chose to work on the same kind of pages the target marks, which supports the target." | Leakage check | An observed overlap between the target and pages someone _chose_ to optimize: part of the overlap is the choosing, so it cannot confirm the target on its own |

**Method:** this cell prints the evidence behind each sentence: the model's Precision@50 against rule B on every test set with its interval; how much more often pages short in month M are short again in month M+1 than pages that were not, inside one size band; and the target rate among pages FlyRank optimized after each pair against the rest.


In [ ]:
# Claim 1: the ranking claim, with the interval and the sealed result.
show_table("Claim 1 evidence: logistic regression against rule B", pd.DataFrame({
    "Precision@50, model": [f"{metric_row(rows, score)['Precision@50']:.2f}" for rows, score in test_sets.values()],
    "Precision@50, rule B": [f"{metric_row(rows, rule_b(rows))['Precision@50']:.2f}" for rows, _ in test_sets.values()],
    "Gap, 95% interval": [f"{low:+.2f} to {high:+.2f}" for low, high in zip(bootstrap["Precision low"], bootstrap["Precision high"], strict=True)],
    "Capture@50, model against rule B": [f"{metric_row(rows, score)['Capture@50']:.1%} against {metric_row(rows, rule_b(rows))['Capture@50']:.1%}"
                                         for rows, score in test_sets.values()],
    "Base rate": [f"{rows['target'].mean():.1%}" for rows, _ in test_sets.values()],
}, index=pd.Index(list(test_sets), name="Test set")))

# Claim 2: persistence, inside one size band so page size cannot explain it.
persistence = []
for name, rows in {**pairs, sealed_name: sealed}.items():
    band = rows[rows["impressions_m"].between(*BAND)]
    short, rest = band.loc[band["shortfall_m"], "target"], band.loc[~band["shortfall_m"], "target"]
    persistence.append({"Pair": name, "Pages short in month M": f"{len(short):,}", "Short again in month M+1": f"{short.mean():.1%}",
                        "Pages not short": f"{len(rest):,}", "Short in month M+1": f"{rest.mean():.1%}", "Ratio": f"{short.mean() / rest.mean():.1f}x"})
show_table("Claim 2 evidence: shortfalls in consecutive months, pages with 1,000 to 9,999 impressions", pd.DataFrame(persistence).set_index("Pair"))

# Claim 3: the target rate among pages FlyRank chose to optimize after the pair, against the rest.
chosen = []
for name, rows in pairs.items():
    later = pd.to_datetime(rows["last_optimized_date"]) > rows["pair_end"]
    chosen.append({"Pair": name, "Optimized later": f"{int(later.sum()):,}", "Real shortfall rate, optimized later": f"{rows.loc[later, 'target'].mean():.1%}",
                   "Real shortfall rate, the rest": f"{rows.loc[~later, 'target'].mean():.1%}",
                   "Median impressions, optimized later": f"{rows.loc[later, 'impressions_m'].median():,.0f}",
                   "Median impressions, the rest": f"{rows.loc[~later, 'impressions_m'].median():,.0f}"})
show_table("Claim 3 evidence: pages FlyRank optimized after the pair", pd.DataFrame(chosen).set_index("Pair"))

## Limitations

- **One sealed month.** The sealed test is a single month pair; it removes the risk of tuning towards the test, but one month cannot show how the model behaves across seasons.
- **Fifty pages per test set.** Every Precision@50 rests on 50 pages, so the intervals above are wide by nature; they are reported so the reader can see how wide.
- **The paper audit is approximate.** The warehouse is a different cut from the paper's 341,701-page snapshot, and the rebuilt health score lacks scroll depth, so section 1 tests the paper's design, not its exact numbers.
- **Segments shrink fast.** Small segments are marked rather than dropped, but their Precision@50 can swing by a single page.
- **Leakage tests catch what they look for.** The audit covers the five standard routes; it cannot rule out an unknown one, which is why the sealed test matters.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.